# Imports

In [2]:
#############################################################################################################################
import pyvisa

# Simulate the `visa` module as an alias for `pyvisa`
import sys

# Create a fake 'visa' module, which is essentially an alias for pyvisa
sys.modules['visa'] = pyvisa

# Optionally, map all attributes from pyvisa to visa (this is technically unnecessary because the alias works)
for attr in dir(pyvisa):
    setattr(sys.modules['visa'], attr, getattr(pyvisa, attr))

#############################################################################################################################


In [3]:
# MAGNET CONTROL
from custom_instruments import daedalusProjField
from pymeasure.adapters import DAQmxAdapter

calib_file = 'C:\\Users\\Ralph Group\\Documents\\Github\\SagnacOperatingSys\\sagnac_control\\calibrations\\sagnac'

magnet = daedalusProjField(DAQmxAdapter('Dev1', ['ao0', 'ai1']),"GPIB::10")
magnet.load_calibration_params(calib_file)

# magnet.set_vector_field(
#     B=0,
#     phi=0, 
#     theta=0)

In [4]:
# LCR METER
from pymeasure.instruments.agilent import Agilent4284A
LCR = Agilent4284A("GPIB::16")
LCR.reset()
# LCR.adapter.connection.timeout = 30000
LCR.frequency


1000.0

In [5]:
# GENERICS
import os
import time

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import plotly.express as px
from IPython.display import display, clear_output



# Basic Comands

In [13]:
calFreqs = [20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 200, 250, 300, 400, 500, 600, 800,
 1000, 1200, 1500, 2000, 2500, 3000, 4000, 5000, 6000, 8000, 10000, 12000, 15000, 
 20000, 25000, 30000, 40000, 50000, 60000, 80000, 100000, 120000, 150000, 200000, 
 250000, 300000, 400000, 500000, 600000, 800000, 1000000]
decadeFreqs = np.logspace(2, 6, 5)
print(decadeFreqs)

[1.e+02 1.e+03 1.e+04 1.e+05 1.e+06]


In [ ]:
LCR.impedance_mode = "RX"
a,b,f = LCR.sweep_measurement(
    'frequency', decadeFreqs     
)                 

In [10]:
magnet.set_vector_field(B=0,phi=0,theta=0)
while magnet.in_motion:
    pass

# Playground

In [12]:
file_path = r"C:\Users\Ralph Group\Documents\Data\Orion\calibration\LCRamrTest\test.csv"

In [14]:
experiment  = "highRes"
B = 0.2
phis = np.arange(-170, 170, 10)
theta = 0

for phi in phis:
    magnet.set_vector_field(B=B,phi=phi,theta=theta)
    while magnet.in_motion:
        pass

    # Record LCR data
    LCR.impedance_mode = "RX"
    a,b,f = LCR.sweep_measurement(
        'frequency', calFreqs     
    )                 
    # Create a DataFrame with frequency (f) and complex data
    df = pd.DataFrame({"experiment":experiment,'phi':phi,'f': f, 'a': a, 'b': b})
    df["phasor"] = df.a + 1j*df.b       
    ## Append to a CSV file (or create a new one if it doesn't exist)
    df.to_csv(file_path, mode='a', header=not os.path.isfile(file_path), index=False)


    dfplt = pd.read_csv(file_path)
    fig = px.scatter(dfplt, x='f', y='a', color='phi')
    clear_output(wait=True)
    display(fig)




ParserError: Error tokenizing data. C error: Expected 5 fields in line 172, saw 6


In [ ]:
dfplt = pd.read_csv(file_path)
fig = px.line(dfplt, x='phi', y='a', color='f')
clear_output(wait=True)
display(fig)

In [12]:
dfplt = pd.read_csv(file_path)
fig = px.line(dfplt, x='f', y='a', color='phi')
clear_output(wait=True)
display(fig)

In [13]:
dfplt = pd.read_csv(file_path)
fig = px.line(dfplt, x='f', y='b', color='phi')
clear_output(wait=True)
display(fig)